## tl;dr

The best corruption-risk evidence comes from **joined event chains**, not from isolated large datasets. For the minimum public MVP, keep the existing four-source boundary: SECOP II contracts, SECOP II processes, contract suspensions, and PACO. These support competition context, repeat-award/concentration context, disruption history, and one exact-NIT documentary signal.

The highest-value next bundle is post-award manipulation evidence—modifications, execution, guarantees, invoices, payment plans, and budget records—but it should remain reviewer-only until historical coverage and calibration are complete. Person-level conflict and corporate-network patterns are also reviewer-only because exact overlap does not prove corrupt intent.

## Context & Methods

This notebook answers: which locally available dataset combinations can support the most useful corruption-risk patterns for co/acc, while preserving a genuinely minimum contest MVP?

The ranking uses five qualitative tests: closeness to a procurement event, exact-key joinability, current local readiness, public-safety constraints, and compute/implementation cost. Materialized signal rows are used only as implementation-readiness evidence. They are overlapping review rows, not unique cases and not estimates of corruption prevalence.

### Key Assumptions

- A red flag or exact overlap is a review lead, never proof of illegality or corrupt intent.
- The public MVP retains one deterministic allowlisted signal: `procurement_sanctioned_supplier_awarded`.
- SECOP offer rows may strengthen evidence, but process-level response counts are enough for the minimum model and avoid loading 42.3 million offer rows into the MVP feature path.
- Reviewer-only sources can be valuable without becoming model inputs or public product scope.
- Local readiness is evaluated against the checked-in curated lake and signal registry as of this analysis.

## Data

Inputs are the current curated signal Parquet directories, `config/signal_registry.yml`, the June 5, 2026 lake reality snapshot, and the high-confidence pattern review in `docs/reviews/high_confidence_missing_corruption_patterns_2026-06-05.md`.

In [1]:
from pathlib import Path
import duckdb
import pandas as pd
import yaml

cwd = Path.cwd().resolve()
repo_root = next(p for p in [cwd, *cwd.parents] if (p / 'config/signal_registry.yml').exists())
curated_root = repo_root / 'lake/curated'
registry_path = repo_root / 'config/signal_registry.yml'
review_path = repo_root / 'docs/reviews/high_confidence_missing_corruption_patterns_2026-06-05.md'
reality_path = repo_root / 'lake/meta/reality/2026-06-05.json'

assert registry_path.exists() and review_path.exists() and reality_path.exists()
repo_root

PosixPath('/Users/ceron/Developer/co-acc')

In [2]:
registry = yaml.safe_load(registry_path.read_text())
registry_signals = {row['id']: row for row in registry['signals']}

con = duckdb.connect()
signal_rows = []
for directory in sorted(curated_root.glob('table=signal_feature_*')):
    parquet_files = list(directory.glob('*.parquet'))
    if not parquet_files:
        continue
    signal_id = directory.name.removeprefix('table=signal_feature_')
    row_count = con.execute(
        'SELECT count(*) FROM read_parquet(?)', [str(directory / '*.parquet')]
    ).fetchone()[0]
    meta = registry_signals.get(signal_id, {})
    signal_rows.append({
        'signal_id': signal_id,
        'materialized_rows': row_count,
        'title': meta.get('title', signal_id),
        'public_safe': bool(meta.get('public_safe', False)),
        'reviewer_only': bool(meta.get('reviewer_only', False)),
        'sources_required': ', '.join(meta.get('sources_required', [])),
    })

signals = pd.DataFrame(signal_rows).sort_values('materialized_rows', ascending=False).reset_index(drop=True)
assert len(signals) == 49, f'Expected 49 materialized signal tables, found {len(signals)}'
signals.head(12)

,signal_id,materialized_rows,title,public_safe,reviewer_only,sources_required
0,procurement_short_bidding_window,90592,Unusually short bidding window,True,False,"secop_ii_processes, secop_offers"
1,procurement_single_bidder_high_value,61542,Single bidder on high-value procurement process,True,False,"secop_offers, secop_ii_processes, secop_ii_con..."
2,public_declaration_supplier_chronology_review_...,25000,Public declarant later appears as supplier,False,True,"asset_disclosures, conflict_disclosures, sigep..."
3,procurement_sanctioned_supplier_awarded,20184,Award to supplier with official sanctions or f...,True,False,"paco_sanctions, fiscal_findings, fiscal_respon..."
4,procurement_role_supplier_same_buyer_review_only,11311,Contract role holder also supplies the same buyer,False,True,secop_ii_contracts
5,procurement_contract_suspensions,10460,Contract with multiple suspensions,True,False,"secop_contract_suspensions, secop_ii_contracts"
6,procurement_repeat_awards_same_supplier,9716,Repeat awards from the same buyer to the same ...,True,False,secop_ii_contracts
7,tvec_item_price_dispersion_review_only,4601,TVEC item line materially above comparable prices,False,True,tvec_orders_consolidated
8,procurement_contract_value_outlier_by_category,2022,Contract value outlier within category,False,True,secop_ii_contracts
9,procurement_related_companies_shared_officer,1482,Supplier cluster sharing officers,False,True,"company_registry_c82u, secop_ii_contracts"


## Results

The following bundles are ranked by decision value, not raw row count. Tier A bundles can directly support the minimum user job or the highest-value next review chain. Tier B bundles are useful enrichments with privacy, coverage, legal-semantics, or scope constraints.

In [3]:
pattern_rows = [
    {
        'priority': 1, 'tier': 'A — MVP core', 'pattern': 'Competition suppression',
        'minimum_sources': 'SECOP II contracts + processes',
        'optional_evidence': 'SECOP II offers',
        'best_examples': 'Very low response count, single bidder, short observed window, repeated co-bidding',
        'public_mvp_role': 'Model context and explanation; offers remain optional',
        'why_ranked_here': 'Directly describes the award process, exact process/contract keys are strong, and aggregate response fields avoid the 42.3M-row offer dependency.',
        'main_caveat': 'Short-window timestamps cover only 8.06% of the bounded 2023+ cohort; absence must not be treated as a normal window.',
    },
    {
        'priority': 2, 'tier': 'A — MVP public signal', 'pattern': 'Sanctioned supplier receives procurement exposure',
        'minimum_sources': 'SECOP II contracts + PACO sanctions',
        'optional_evidence': 'SECOP sanctions + SIRI + fiscal findings/responsibility',
        'best_examples': 'Exact-NIT sanction/finding record linked to supplier awards',
        'public_mvp_role': 'The one required allowlisted deterministic signal',
        'why_ranked_here': 'Clear cross-source story, exact company identity, official evidence links, and 20,184 materialized PACO/fiscal/sanction overlap rows.',
        'main_caveat': 'A sanction or finding overlap does not establish current ineligibility, contract illegality, or corrupt intent.',
    },
    {
        'priority': 3, 'tier': 'A — Supporting context', 'pattern': 'Repeat awards and supplier concentration',
        'minimum_sources': 'SECOP II contracts',
        'optional_evidence': 'SECOP I historical contracts',
        'best_examples': 'Repeated same buyer/supplier awards, concentrated buyer spend, value outliers',
        'public_mvp_role': 'Feature explanation and prioritization context, not a standalone allegation',
        'why_ranked_here': 'Broad coverage, no extra ingest dependency, and useful context for both anomaly scoring and human review.',
        'main_caveat': 'Concentration can be legitimate in specialized or thin markets and needs buyer/category context.',
    },
    {
        'priority': 4, 'tier': 'A — Best next bundle', 'pattern': 'Post-award manipulation or execution failure chain',
        'minimum_sources': 'Contracts + suspensions + modifications + execution',
        'optional_evidence': 'Guarantees + invoices + payment plans + CDP/commitments/budget items',
        'best_examples': 'Modification ladder, repeated suspensions, high advance, delayed execution, invoice/payment mismatch',
        'public_mvp_role': 'Suspensions are MVP features; the full chain stays reviewer-only until backfilled',
        'why_ranked_here': 'Multiple events on the same exact contract create the strongest explanatory chain after award.',
        'main_caveat': 'Modification and execution history is locally incomplete; legality and guarantee validity require source-file review.',
    },
    {
        'priority': 5, 'tier': 'B — Reviewer only', 'pattern': 'Corporate network and related bidders',
        'minimum_sources': 'Contracts + offers + RUES company registry',
        'optional_evidence': 'SECOP supplier registry + branch/address records',
        'best_examples': 'Shared legal representative, related bidders in one process, same-buyer company cluster',
        'public_mvp_role': 'Do not expose in the minimum public product',
        'why_ranked_here': 'Exact NIT and representative joins can reveal hidden competition structure.',
        'main_caveat': 'RUES coverage and role-date semantics need calibration; common representatives and business groups can be legitimate.',
    },
    {
        'priority': 6, 'tier': 'B — Reviewer only', 'pattern': 'Conflict, insider, or political connection',
        'minimum_sources': 'Contracts + declarations/SIGEP or Cuentas Claras',
        'optional_evidence': 'RUES legal representatives',
        'best_examples': 'Same-buyer role holder becomes supplier, donor-to-supplier chronology, declared interest/company bridge',
        'public_mvp_role': 'Reviewer queue only',
        'why_ranked_here': 'Potentially high narrative value and exact document joins, with several implemented queues.',
        'main_caveat': 'Person-level privacy, stale 2019/2022 windows, missing legal thresholds, and high legitimate-overlap risk.',
    },
    {
        'priority': 7, 'tier': 'B — Reviewer only', 'pattern': 'Framework-agreement price dispersion',
        'minimum_sources': 'TVEC item purchases',
        'optional_evidence': 'Contracts + supplier sanctions/concentration',
        'best_examples': 'Same item/unit/year price at p95 and at least twice the median',
        'public_mvp_role': 'Reviewer queue only',
        'why_ranked_here': 'Item-level price comparators are intuitive and 4,601 review rows are already materialized.',
        'main_caveat': 'Item normalization, quality, geography, delivery terms, and framework-specific conditions can explain price differences.',
    },
    {
        'priority': 8, 'tier': 'B — Reviewer only', 'pattern': 'Project spending versus delivery gap',
        'minimum_sources': 'Contracts + BPIN links + SGR execution/projects',
        'optional_evidence': 'DNP executors, locations, and beneficiary context',
        'best_examples': 'High linked procurement with low physical/financial execution or weak beneficiary delivery evidence',
        'public_mvp_role': 'Reviewer queue for the SGR/project subset',
        'why_ranked_here': 'Strong project-level narrative when exact BPIN links exist.',
        'main_caveat': 'Sparse whole-contract coverage and project timing make this a subset analysis, not a general MVP feature.',
    },
]
patterns = pd.DataFrame(pattern_rows)
patterns[['priority', 'tier', 'pattern', 'minimum_sources', 'public_mvp_role', 'main_caveat']]

,priority,tier,pattern,minimum_sources,public_mvp_role,main_caveat
0,1,A — MVP core,Competition suppression,SECOP II contracts + processes,Model context and explanation; offers remain o...,Short-window timestamps cover only 8.06% of th...
1,2,A — MVP public signal,Sanctioned supplier receives procurement exposure,SECOP II contracts + PACO sanctions,The one required allowlisted deterministic signal,A sanction or finding overlap does not establi...
2,3,A — Supporting context,Repeat awards and supplier concentration,SECOP II contracts,Feature explanation and prioritization context...,Concentration can be legitimate in specialized...
3,4,A — Best next bundle,Post-award manipulation or execution failure c...,Contracts + suspensions + modifications + exec...,Suspensions are MVP features; the full chain s...,Modification and execution history is locally ...
4,5,B — Reviewer only,Corporate network and related bidders,Contracts + offers + RUES company registry,Do not expose in the minimum public product,RUES coverage and role-date semantics need cal...
5,6,B — Reviewer only,"Conflict, insider, or political connection",Contracts + declarations/SIGEP or Cuentas Claras,Reviewer queue only,"Person-level privacy, stale 2019/2022 windows,..."
6,7,B — Reviewer only,Framework-agreement price dispersion,TVEC item purchases,Reviewer queue only,"Item normalization, quality, geography, delive..."
7,8,B — Reviewer only,Project spending versus delivery gap,Contracts + BPIN links + SGR execution/projects,Reviewer queue for the SGR/project subset,Sparse whole-contract coverage and project tim...


In [4]:
family_members = {
    'Competition suppression': [
        'procurement_short_bidding_window', 'procurement_single_bidder_high_value',
        'procurement_cartel_risk_cobidding', 'procurement_offers_competition_drop',
        'procurement_related_bidders_same_process_review_only',
    ],
    'Sanction chronology': [
        'procurement_sanctioned_supplier_awarded', 'procurement_secop_sanction_later_awards_review_only',
        'fiscal_procurement_chronology_review_only', 'siri_antecedent_procurement_chronology_review_only',
    ],
    'Repeat awards and concentration': [
        'procurement_repeat_awards_same_supplier', 'procurement_supplier_concentration_across_entities',
        'procurement_buyer_supplier_network_density', 'procurement_contract_value_outlier_by_category',
    ],
    'Post-award and payment chain': [
        'procurement_contract_suspensions', 'procurement_large_modifications',
        'procurement_contract_modification_ladder_review_only', 'procurement_contract_execution_delay',
        'procurement_payment_plan_anomalies', 'procurement_budget_chain_reconciliation_review_only',
        'procurement_invoice_budget_reconciliation_review_only',
        'procurement_payment_plan_reconciliation_review_only',
        'procurement_guarantee_advance_execution_chain',
        'procurement_guarantee_policy_reuse_review_only',
    ],
    'Conflict and insider links': [
        'public_declaration_supplier_chronology_review_only',
        'procurement_role_supplier_same_buyer_review_only',
        'procurement_politically_exposed_position_supplier_overlap',
        'procurement_public_servant_conflict_disclosure_overlap',
        'public_declaration_company_bridge_current_risk_review_only',
        'cuentas_claras_donor_supplier_overlap', 'cuentas_claras_donor_ineligibility_review',
    ],
    'Corporate network and identity': [
        'procurement_related_companies_shared_officer', 'rues_supplier_capacity_status_review_only',
        'procurement_cross_source_identity_inconsistency',
        'procurement_shared_representative_same_buyer_cluster_review_only',
        'secop_i_legacy_supplier_current_risk_review_only',
        'secop_i_legacy_representative_current_risk_review_only',
    ],
    'TVEC price and capture': ['tvec_item_price_dispersion_review_only', 'tvec_multi_entity_capture'],
    'Project delivery gap': [
        'project_bpin_procurement_overlap', 'project_regalias_execution_procurement_overlap',
        'sgr_ocad_executor_capacity_gap', 'dnp_sgr_beneficiary_delivery_gap_review_only',
    ],
}

count_lookup = signals.set_index('signal_id')['materialized_rows'].to_dict()
missing_group_signals = sorted({s for members in family_members.values() for s in members} - set(count_lookup))
assert not missing_group_signals, missing_group_signals
family_support = pd.DataFrame([
    {
        'pattern_family': family,
        'materialized_signal_rows': sum(count_lookup[s] for s in members),
        'signal_count': len(members),
        'interpretation': 'Overlapping signal rows; implementation scale only',
    }
    for family, members in family_members.items()
]).sort_values('materialized_signal_rows', ascending=False).reset_index(drop=True)
family_support

,pattern_family,materialized_signal_rows,signal_count,interpretation
0,Competition suppression,153029,5,Overlapping signal rows; implementation scale ...
1,Conflict and insider links,37573,7,Overlapping signal rows; implementation scale ...
2,Sanction chronology,21511,4,Overlapping signal rows; implementation scale ...
3,Post-award and payment chain,15141,10,Overlapping signal rows; implementation scale ...
4,Repeat awards and concentration,12599,4,Overlapping signal rows; implementation scale ...
5,TVEC price and capture,4688,2,Overlapping signal rows; implementation scale ...
6,Corporate network and identity,3312,6,Overlapping signal rows; implementation scale ...
7,Project delivery gap,815,4,Overlapping signal rows; implementation scale ...


In [5]:
# Decision-impact spot checks
import json

assert count_lookup['procurement_sanctioned_supplier_awarded'] == 20_184
assert count_lookup['procurement_short_bidding_window'] == 90_592
assert count_lookup['procurement_single_bidder_high_value'] == 61_542
assert count_lookup['procurement_contract_suspensions'] == 10_460
assert count_lookup['procurement_repeat_awards_same_supplier'] == 9_716
assert count_lookup['procurement_large_modifications'] == 582
assert signals['materialized_rows'].ge(0).all()
assert patterns['priority'].is_unique and patterns['priority'].tolist() == list(range(1, 9))

metrics_path = repo_root / 'docs/analysis/corruption_pattern_dataset_metrics.json'
metrics = json.loads(metrics_path.read_text())
assert metrics['materialized_signal_tables'] == len(signals)
computed_family_totals = dict(zip(family_support['pattern_family'], family_support['materialized_signal_rows']))
saved_family_totals = {row['pattern_family']: row['materialized_signal_rows'] for row in metrics['family_support']}
assert saved_family_totals == computed_family_totals
assert [row['pattern'] for row in metrics['pattern_ranking']] == patterns['pattern'].tolist()

validation_summary = {
    'materialized_signal_tables': len(signals),
    'ranked_pattern_bundles': len(patterns),
    'public_mvp_required_signal': 'procurement_sanctioned_supplier_awarded',
    'highest_value_next_source': 'SECOP contract modifications after complete 2023+ backfill',
    'row_count_warning': 'Signal rows overlap and may be capped; they are not unique cases or prevalence.',
}
validation_summary

{'materialized_signal_tables': 49,
 'ranked_pattern_bundles': 8,
 'public_mvp_required_signal': 'procurement_sanctioned_supplier_awarded',
 'highest_value_next_source': 'SECOP contract modifications after complete 2023+ backfill',
 'row_count_warning': 'Signal rows overlap and may be capped; they are not unique cases or prevalence.'}

## Takeaways

1. **Do not add datasets merely to increase the count.** Contracts plus process aggregates already support the most scalable competition patterns.
2. **Keep PACO because it supplies the clearest minimum cross-source proof point.** The public product should expose documentary overlap, not claim corruption.
3. **Use suspensions now and promote modifications only after backfill.** The full post-award chain is the best next reviewer capability, not a launch dependency.
4. **Keep offers, RUES, declarations, TVEC, BPIN/SGR, guarantees, invoices, and payments outside the core model.** They can enrich evidence or reviewer queues without expanding the public MVP or adding a second model.
5. **Preserve one public signal at launch.** The other strong patterns should remain explanations, internal review queues, or sequenced extensions until their coverage, privacy, and legal-semantics gates pass.